In [49]:
import polars as pl
import numpy as np
import re

In [50]:
path = "C:\\Users\\Arnold\\OneDrive\\Desktop\\CAPSTONE PROJECT\\farming_risk_regions\\data\\interim\\weather_data\\all_locations_weather_data.csv"
weather_data = pl.read_csv(path, try_parse_dates=True)

In [51]:
# Check the schema and first few rows
print(weather_data.schema)
print("\nFirst few rows:")
weather_data.head()

Schema({'location': String, 'date': Date, 'lat_center': Float64, 'lon_center': Float64, 'lat_min': Float64, 'lat_max': Float64, 'lon_min': Float64, 'lon_max': Float64, 'ALLSKY_SFC_SW_DWN': Float64, 'PRECTOTCORR': Float64, 'RH2M': Float64, 'T2M_MAX': Float64, 'T2M_MIN': Float64, 'WS10M': Float64})

First few rows:


location,date,lat_center,lon_center,lat_min,lat_max,lon_min,lon_max,ALLSKY_SFC_SW_DWN,PRECTOTCORR,RH2M,T2M_MAX,T2M_MIN,WS10M
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Story_County_IA""",2014-01-01,42.025,-93.5,41.85,42.2,-93.81,-93.19,3.42,1.2,77.96,-13.07,-17.98,6.41
"""Story_County_IA""",2014-01-02,42.025,-93.5,41.85,42.2,-93.81,-93.19,8.17,0.0,77.08,-11.63,-18.55,4.37
"""Story_County_IA""",2014-01-03,42.025,-93.5,41.85,42.2,-93.81,-93.19,4.86,0.0,84.33,-2.26,-18.78,8.83
"""Story_County_IA""",2014-01-04,42.025,-93.5,41.85,42.2,-93.81,-93.19,3.2,0.01,72.45,-1.7,-12.34,8.15
"""Story_County_IA""",2014-01-05,42.025,-93.5,41.85,42.2,-93.81,-93.19,6.04,0.0,56.36,-13.14,-23.8,10.53


In [52]:
## 
def calculate_weather_variables(dataframe):
    '''
    Function to calculate additional weather variables from the existing data using Polars.

    PRECTOTCORR - Precipitation:
        - ppt_intensity: rainfall intensity (total/rainy days)
        - ppt_variance: precipitation variability
        - rainy_days: number of days with precipitation > 1mm
        
    T2M_MAX, T2M_MIN - Temperature:
        - tmin: mean minimum temperature
        - temp_range: mean diurnal temperature range (Tmax - Tmin)
        - gdd: growing degree days (base 10°C)
        - frost_days: days with Tmin < 0°C
        - stress_days: days with extreme heat (Tmax > 35°C)
        
    WS10M - Wind Speed:
        - wind_mean: average wind speed
        - wind_max: maximum wind speed
        - high_wind_days: days with wind > 10 m/s
        
    ALLSKY_SFC_SW_DWN - Solar Radiation:
        - solar_mean: average solar radiation
        - solar_sum: cumulative solar radiation
        - low_light_days: days with radiation < threshold
        
    RH2M - Relative Humidity:
        - rh_mean: average relative humidity
        - high_humidity_days: days with RH > 90%
        - vpd: vapor pressure deficit (evaporative demand)    
    '''
    
    # Add year and month columns for grouping
    df = dataframe.with_columns([
        pl.col('date').dt.year().alias('year'),
        pl.col('date').dt.month().alias('month')
    ])
    
    # Create monthly aggregated features
    monthly_weather = df.group_by(['location', 'year', 'month']).agg([
        # ===== PRECIPITATION VARIABLES =====
        pl.col('PRECTOTCORR').sum().alias('ppt_total'),
        (pl.col('PRECTOTCORR') > 1).sum().alias('rainy_days'),
        pl.col('PRECTOTCORR').std().alias('ppt_variance'),
        
        # ===== TEMPERATURE VARIABLES =====
        pl.col('T2M_MAX').mean().alias('tmax'),
        pl.col('T2M_MIN').mean().alias('tmin'),
        (pl.col('T2M_MAX') - pl.col('T2M_MIN')).mean().alias('temp_range'),
        
        # Extreme temperature days
        (pl.col('T2M_MAX') > 30).sum().alias('tmax30'),
        (pl.col('T2M_MAX') > 35).sum().alias('tmax35'),
        (pl.col('T2M_MIN') < 0).sum().alias('tmin0'),
        
        # Growing Degree Days (GDD) - base 10°C
        (((pl.col('T2M_MAX') + pl.col('T2M_MIN')) / 2 - 10).clip(lower_bound=0)).sum().alias('gdd'),
        
        # ===== WIND VARIABLES =====
        pl.col('WS10M').mean().alias('wind_mean'),
        pl.col('WS10M').max().alias('wind_max'),
        pl.col('WS10M').std().alias('wind_variance'),
        (pl.col('WS10M') > 10).sum().alias('high_wind_days'),
        
        # ===== SOLAR RADIATION VARIABLES =====
        pl.col('ALLSKY_SFC_SW_DWN').mean().alias('solar_mean'),
        pl.col('ALLSKY_SFC_SW_DWN').sum().alias('solar_sum'),
        pl.col('ALLSKY_SFC_SW_DWN').std().alias('solar_variance'),
        (pl.col('ALLSKY_SFC_SW_DWN') < 115.7).sum().alias('low_light_days'),
        
        # ===== HUMIDITY VARIABLES =====
        pl.col('RH2M').mean().alias('rh_mean'),
        pl.col('RH2M').min().alias('rh_min'),
        pl.col('RH2M').max().alias('rh_max'),
        (pl.col('RH2M') > 90).sum().alias('high_humidity_days'),
        (pl.col('RH2M') < 30).sum().alias('low_humidity_days'),
        
        # ===== COMPOUND INDICES =====
        # Heat-humidity stress
        ((pl.col('T2M_MAX') > 30) & (pl.col('RH2M') > 70)).sum().alias('heat_humidity_stress_days'),
        
        # Drought stress
        ((pl.col('PRECTOTCORR') < 2) & (pl.col('T2M_MAX') > 30) & (pl.col('RH2M') < 40)).sum().alias('drought_stress_days'),
        
        # Storm potential
        ((pl.col('PRECTOTCORR') > 20) & (pl.col('WS10M') > 8)).sum().alias('storm_potential_days'),
        
        # Cloudy warm days
        ((pl.col('T2M_MAX') > 20) & (pl.col('ALLSKY_SFC_SW_DWN') < 150)).sum().alias('cloudy_warm_days'),
    ])
    
    # Calculate derived variables that need the aggregated data
    monthly_weather = monthly_weather.with_columns([
        # Precipitation intensity (total rainfall / rainy days)
        (pl.col('ppt_total') / pl.when(pl.col('rainy_days') > 0).then(pl.col('rainy_days')).otherwise(1)).alias('ppt_intensity'),
        
        # Vapor Pressure Deficit (VPD)
        # VPD = (1 - RH/100) * es(T), where es is saturation vapor pressure
        # Saturation vapor pressure: es = 0.6108 * exp((17.27 * T) / (T + 237.3))
        ((1 - pl.col('rh_mean') / 100) * 
         (0.6108 * (17.27 * (pl.col('tmax') + pl.col('tmin')) / 2 / 
          ((pl.col('tmax') + pl.col('tmin')) / 2 + 237.3)).exp())).alias('vpd_mean')
    ])
    
    return monthly_weather

In [53]:
monthly_weather = calculate_weather_variables(weather_data)
monthly_weather

location,year,month,ppt_total,rainy_days,ppt_variance,tmax,tmin,temp_range,tmax30,tmax35,tmin0,gdd,wind_mean,wind_max,wind_variance,high_wind_days,solar_mean,solar_sum,solar_variance,low_light_days,rh_mean,rh_min,rh_max,high_humidity_days,low_humidity_days,heat_humidity_stress_days,drought_stress_days,storm_potential_days,cloudy_warm_days,ppt_intensity,vpd_mean
str,i32,i8,f64,u32,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64
"""Pottawattamie_County_IA""",2023,2,45.11,5,4.678099,5.078929,-6.7,11.778929,0,0,27,0.0,5.7025,9.21,1.671348,0,10.476071,293.33,3.940904,28,72.836786,49.33,92.18,2,0,0,0,0,0,9.022,0.156377
"""Pottawattamie_County_IA""",2019,10,113.16,9,8.396917,13.594516,2.505484,11.089032,0,0,12,35.865,5.281935,8.57,1.805439,0,11.52,357.12,4.522288,31,75.623226,64.12,92.71,1,0,0,0,0,4,12.573333,0.262398
"""Pottawattamie_County_IA""",2014,7,66.07,8,7.741127,27.037097,15.549677,11.487419,7,0,0,350.095,3.809032,6.76,1.514533,0,23.087419,715.71,4.509607,31,73.316129,58.37,89.59,0,0,7,0,0,31,8.25875,0.675683
"""Dawson_County_NE""",2024,9,5.6,2,0.421388,30.059667,15.116667,14.943,17,1,0,377.645,5.106667,9.98,1.89507,0,19.206667,576.2,3.726335,30,47.712333,31.91,66.85,0,0,0,3,0,30,2.8,1.432793
"""Dawson_County_NE""",2020,3,56.78,8,3.682706,11.757419,-1.337742,13.095161,0,0,18,10.55,5.566774,9.17,1.557696,0,13.553548,420.16,6.251031,31,72.086452,51.37,94.29,1,0,0,0,0,6,7.0975,0.247083
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Livingston_County_IL""",2021,10,211.44,14,14.317635,19.624516,9.837097,9.787419,1,0,0,157.835,4.416129,7.36,1.49105,0,9.240323,286.45,4.233695,31,80.449677,65.12,94.0,3,0,0,0,0,16,15.102857,0.327666
"""Champaign_County_IL""",2021,2,43.53,6,4.369319,-1.294286,-11.679643,10.385357,0,0,26,0.0,5.234286,9.19,1.764561,0,11.484643,321.57,3.669866,28,87.957143,75.83,96.4,15,0,0,0,0,0,7.255,0.045273
"""Champaign_County_IL""",2023,10,97.91,11,6.21597,19.585161,7.882903,11.702258,4,0,2,137.365,4.780968,6.9,1.349838,0,10.724839,332.47,4.829041,31,69.966129,53.99,84.39,0,0,0,0,0,13,8.900909,0.471899


In [54]:
def calculate_anomalies(weather_df: pl.DataFrame, vars: list[str] | None = None, group_cols: list[str] | None = None) -> pl.DataFrame:
    """
    Transform weather variables into standardized anomalies (z-scores) per group.
    x_AN = (x - mean_group) / std_group

    """
    group_cols = ['location', 'month']

    preferred = [
        'ppt_total', 'ppt_intensity', 'ppt_variance',
        'tmax', 'tmin', 'temp_range', 'gdd',
        'tmax30', 'tmax35', 'tmin0',
        'wind_mean', 'wind_max', 'wind_variance', 'high_wind_days',
        'solar_mean', 'solar_sum', 'solar_variance', 'low_light_days',
        'rh_mean', 'rh_min', 'rh_max', 'high_humidity_days', 'low_humidity_days',
        'heat_humidity_stress_days', 'drought_stress_days', 'storm_potential_days', 'cloudy_warm_days',
        'vpd_mean'
    ]
    if vars is None:
        vars = [v for v in preferred if v in weather_df.columns]
    else:
        vars = [v for v in vars if v in weather_df.columns]

    if not vars:
        raise ValueError("No valid variables found to compute anomalies; please pass 'vars' that exist in the DataFrame.")

    # Compute group-wise stats for all variables in a single aggregation
    agg_exprs = []
    for v in vars:
        agg_exprs.append(pl.col(v).mean().alias(f"{v}__mean"))
        agg_exprs.append(pl.col(v).std().alias(f"{v}__std"))

    stats = weather_df.group_by(group_cols).agg(agg_exprs)

    # Join stats back and compute anomalies
    df = weather_df.join(stats, on=group_cols, how='left')

    an_exprs = []
    for v in vars:
        mean_col = pl.col(f"{v}__mean")
        std_col = pl.col(f"{v}__std")
        an_exprs.append(
            pl.when(std_col.fill_null(0).abs() > 0)
            .then((pl.col(v) - mean_col) / std_col)
            .otherwise(pl.lit(0.0))
            .alias(f"{v}_AN")
        )
    df = df.with_columns(an_exprs)

    # Drop helper columns 
    drop_cols = [f"{v}__mean" for v in vars] + [f"{v}__std" for v in vars]
    df = df.drop(drop_cols)

    return df

In [55]:
# Compute anomalies using Polars (grouped by location and month)
anomalies = calculate_anomalies(monthly_weather)
anomalies.head()

location,year,month,ppt_total,rainy_days,ppt_variance,tmax,tmin,temp_range,tmax30,tmax35,tmin0,gdd,wind_mean,wind_max,wind_variance,high_wind_days,solar_mean,solar_sum,solar_variance,low_light_days,rh_mean,rh_min,rh_max,high_humidity_days,low_humidity_days,heat_humidity_stress_days,drought_stress_days,storm_potential_days,cloudy_warm_days,ppt_intensity,vpd_mean,ppt_total_AN,ppt_intensity_AN,ppt_variance_AN,tmax_AN,tmin_AN,temp_range_AN,gdd_AN,tmax30_AN,tmax35_AN,tmin0_AN,wind_mean_AN,wind_max_AN,wind_variance_AN,high_wind_days_AN,solar_mean_AN,solar_sum_AN,solar_variance_AN,low_light_days_AN,rh_mean_AN,rh_min_AN,rh_max_AN,high_humidity_days_AN,low_humidity_days_AN,heat_humidity_stress_days_AN,drought_stress_days_AN,storm_potential_days_AN,cloudy_warm_days_AN,vpd_mean_AN
str,i32,i8,f64,u32,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Pottawattamie_County_IA""",2023,2,45.11,5,4.678099,5.078929,-6.7,11.778929,0,0,27,0.0,5.7025,9.21,1.671348,0,10.476071,293.33,3.940904,28,72.836786,49.33,92.18,2,0,0,0,0,0,9.022,0.156377,1.602094,1.401642,2.050049,0.624156,0.562005,0.687458,-0.430643,0.0,0.0,0.113067,0.79576,-0.614319,-0.550638,-0.661107,-0.807768,-0.877266,0.759369,-0.583874,-0.522651,-0.959313,-0.146763,-0.343797,0.0,0.0,0.0,0.0,-0.301511,0.56957
"""Pottawattamie_County_IA""",2019,10,113.16,9,8.396917,13.594516,2.505484,11.089032,0,0,12,35.865,5.281935,8.57,1.805439,0,11.52,357.12,4.522288,31,75.623226,64.12,92.71,1,0,0,0,0,4,12.573333,0.262398,1.099939,0.278926,1.000077,-1.540222,-1.610142,-0.78007,-1.391006,-0.619687,0.0,1.971679,0.891925,-0.526612,0.308559,-0.301511,-0.359371,-0.359371,0.520184,0.0,0.690159,0.971076,0.540556,-0.175382,0.0,0.0,-0.301511,-0.301511,-1.411588,-0.896625
"""Pottawattamie_County_IA""",2014,7,66.07,8,7.741127,27.037097,15.549677,11.487419,7,0,0,350.095,3.809032,6.76,1.514533,0,23.087419,715.71,4.509607,31,73.316129,58.37,89.59,0,0,7,0,0,31,8.25875,0.675683,-0.635174,0.502122,1.405602,-1.684229,-2.411544,-0.028304,-2.126234,-1.176666,-0.976937,0.0,0.796204,0.472567,1.32964,0.0,-0.346961,-0.346961,-1.23915,0.0,0.53126,0.342002,0.661594,-0.449467,0.0,0.754434,0.0,-0.301511,0.301511,-0.987505
"""Dawson_County_NE""",2024,9,5.6,2,0.421388,30.059667,15.116667,14.943,17,1,0,377.645,5.106667,9.98,1.89507,0,19.206667,576.2,3.726335,30,47.712333,31.91,66.85,0,0,0,3,0,30,2.8,1.432793,-1.941319,-1.624003,-1.951427,1.319326,0.920771,0.989327,1.24805,1.57611,-0.456435,0.0,0.555095,1.409471,1.475385,0.0,1.673073,1.673073,-1.303873,0.0,-1.432206,-1.144068,-2.178151,0.0,0.0,-0.57172,0.793329,0.0,0.911414,1.647483
"""Dawson_County_NE""",2020,3,56.78,8,3.682706,11.757419,-1.337742,13.095161,0,0,18,10.55,5.566774,9.17,1.557696,0,13.553548,420.16,6.251031,31,72.086452,51.37,94.29,1,0,0,0,0,6,7.0975,0.247083,0.279972,-0.082617,0.015553,0.067337,0.452027,-0.559504,-0.169193,0.0,0.0,-0.609644,-0.544531,-1.357074,-1.589233,-1.556998,-1.535932,-1.535932,1.423033,0.0,1.290727,1.023395,1.0846,0.191977,0.0,0.0,0.0,-0.421741,0.709255,-0.849359


In [56]:
def create_lagged_features(df: pl.DataFrame, lag_months: list[int]) -> pl.DataFrame:

    lagged_dataframe= df.clone()
    vars = [col for col in df.columns if col.endswith('_AN')]

    for var in vars:
        for lag in lag_months:
            lagged_dataframe = lagged_dataframe.with_columns(
                df.select([pl.col(var).shift(lag).over("location").alias(f"{var}_lag{lag}")])
            )

    return lagged_dataframe

In [57]:
lagged_weather = create_lagged_features(anomalies, lag_months=[1, 2])
lagged_weather = lagged_weather.drop_nulls()
lagged_weather.head()

location,year,month,ppt_total,rainy_days,ppt_variance,tmax,tmin,temp_range,tmax30,tmax35,tmin0,gdd,wind_mean,wind_max,wind_variance,high_wind_days,solar_mean,solar_sum,solar_variance,low_light_days,rh_mean,rh_min,rh_max,high_humidity_days,low_humidity_days,heat_humidity_stress_days,drought_stress_days,storm_potential_days,cloudy_warm_days,ppt_intensity,vpd_mean,ppt_total_AN,ppt_intensity_AN,ppt_variance_AN,tmax_AN,tmin_AN,…,tmin0_AN_lag2,wind_mean_AN_lag1,wind_mean_AN_lag2,wind_max_AN_lag1,wind_max_AN_lag2,wind_variance_AN_lag1,wind_variance_AN_lag2,high_wind_days_AN_lag1,high_wind_days_AN_lag2,solar_mean_AN_lag1,solar_mean_AN_lag2,solar_sum_AN_lag1,solar_sum_AN_lag2,solar_variance_AN_lag1,solar_variance_AN_lag2,low_light_days_AN_lag1,low_light_days_AN_lag2,rh_mean_AN_lag1,rh_mean_AN_lag2,rh_min_AN_lag1,rh_min_AN_lag2,rh_max_AN_lag1,rh_max_AN_lag2,high_humidity_days_AN_lag1,high_humidity_days_AN_lag2,low_humidity_days_AN_lag1,low_humidity_days_AN_lag2,heat_humidity_stress_days_AN_lag1,heat_humidity_stress_days_AN_lag2,drought_stress_days_AN_lag1,drought_stress_days_AN_lag2,storm_potential_days_AN_lag1,storm_potential_days_AN_lag2,cloudy_warm_days_AN_lag1,cloudy_warm_days_AN_lag2,vpd_mean_AN_lag1,vpd_mean_AN_lag2
str,i32,i8,f64,u32,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Pottawattamie_County_IA""",2014,7,66.07,8,7.741127,27.037097,15.549677,11.487419,7,0,0,350.095,3.809032,6.76,1.514533,0,23.087419,715.71,4.509607,31,73.316129,58.37,89.59,0,0,7,0,0,31,8.25875,0.675683,-0.635174,0.502122,1.405602,-1.684229,-2.411544,…,0.113067,0.891925,0.79576,-0.526612,-0.614319,0.308559,-0.550638,-0.301511,-0.661107,-0.359371,-0.807768,-0.359371,-0.877266,0.520184,0.759369,0.0,-0.583874,0.690159,-0.522651,0.971076,-0.959313,0.540556,-0.146763,-0.175382,-0.343797,0.0,0.0,0.0,0.0,-0.301511,0.0,-0.301511,0.0,-1.411588,-0.301511,-0.896625,0.56957
"""Renville_County_MN""",2016,11,38.15,8,2.957666,10.483667,0.480667,10.003,0,0,14,13.74,4.890333,12.55,2.163842,1,7.528667,225.86,3.018156,30,80.379667,70.53,95.58,6,0,0,0,0,2,4.76875,0.176997,0.253707,-0.283365,0.139449,1.306462,1.407077,…,0.0,-0.527662,0.367306,-0.74176,0.766814,-0.890692,1.061671,-0.301511,0.80403,0.736676,-0.441418,0.736676,-0.441418,0.717877,0.314519,0.0,0.0,-1.480872,-0.689707,-1.167655,-0.678929,-1.295709,-0.699574,0.0,-0.771421,0.0,0.0,0.94388,0.0,0.0,0.0,-0.661107,0.0,-0.330902,0.0,0.758847,0.624329
"""Pottawattamie_County_IA""",2018,4,20.21,3,1.678066,12.547333,-0.508333,13.055667,0,0,17,32.545,5.633667,11.23,2.251984,2,17.227667,516.83,7.79867,30,67.539,52.14,86.11,0,0,0,0,0,6,6.736667,0.303956,-1.253077,-0.164464,-1.103373,-2.445937,-2.269323,…,1.971679,0.796204,0.891925,0.472567,-0.526612,1.32964,0.308559,0.0,-0.301511,-0.346961,-0.359371,-0.346961,-0.359371,-1.23915,0.520184,0.0,0.0,0.53126,0.690159,0.342002,0.971076,0.661594,0.540556,-0.449467,-0.175382,0.0,0.0,0.754434,0.0,0.0,-0.301511,-0.301511,-0.301511,0.301511,-1.411588,-0.987505,-0.896625
"""Champaign_County_IL""",2019,9,82.1,10,4.629088,28.643333,16.833333,11.81,12,0,0,382.15,3.831667,6.69,1.318257,0,15.768667,473.06,4.794204,30,72.775667,57.51,86.82,0,0,7,0,0,30,8.21,0.752829,0.319939,-0.377972,-0.645944,1.189974,1.744997,…,-0.476731,-0.813278,0.410277,-1.85685,0.49994,-1.587781,1.33175,0.0,-0.449467,1.816255,-1.321049,1.816255,-1.321049,-1.686762,1.019129,0.0,0.0,-0.141175,1.692854,-0.850143,0.62701,0.206034,0.68209,0.0,0.568149,0.0,0.0,-0.393369,0.0,0.0,0.0,0.0,1.556998,0.0,0.028339,-0.088304,-0.87681
"""Story_County_IA""",2018,5,102.11,15,5.211043,26.19871,14.191935,12.006774,6,2,0,316.055,4.009032,7.11,1.397892,0,20.592903,638.38,7.684696,31,73.282903,55.05,89.65,0,0,2,0,0,28,6.807333,0.632314,-0.215643,-0.753367,-0.336808,2.042113,2.489856,…,0.0,1.18009,-1.520694,-0.076793

In [58]:
insurance_df = pl.read_csv("C:\\Users\\Arnold\\OneDrive\\Desktop\\CAPSTONE PROJECT\\farming_risk_regions\\data\\interim\\insurance_data\\insurance_data_cleaned.csv", try_parse_dates=True)
insurance_df.tail()

commodity_year,state_code,state_abbreviation,county_code,county_name,commodity_code,commodity_name,insurance_plan_code,insurance_plan_abbreviation,coverage_category,delivery_type,coverage_level,policies_sold_count,policies_earning_premium_count,policies_indemnified_count,units_earning_premium_count,units_indemnified_count,quantity_type,net_reported_quantity,endorsed_companion_acres,liability_amount,total_premium_amount,subsidy_amount,state_private_subsidy,additional_subsidy,efa_premium_discount,indemnity_amount,loss_ratio,county_state
i64,i64,str,i64,str,i64,str,i64,str,str,str,f64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,str
2025,56,"""WY""",45,"""Weston """,9999,"""All Other Crops """,13,"""RI ""","""A ""","""RBUP""",0.9,7,6,1,20,1,"""Acres """,92291,0,1101710,197664,100810,0,0,0,5350,0.03,"""Weston,WY"""
2025,56,"""WY""",45,"""Weston """,9999,"""All Other Crops """,2,"""RP ""","""A ""","""RBUP""",0.65,1,1,1,1,1,"""Acres """,40,0,3912,621,366,0,0,0,3508,5.65,"""Weston,WY"""
2025,56,"""WY""",45,"""Weston """,9999,"""All Other Crops """,1,"""YP ""","""A ""","""RBUP""",0.5,1,0,0,0,0,"""Acres """,0,0,0,0,0,0,0,0,0,0.0,"""Weston,WY"""
2025,56,"""WY""",45,"""Weston """,9999,"""All Other Crops """,1,"""YP ""","""A ""","""RBUP""",0.65,3,1,0,1,0,"""Acres """,30,0,1143,584,345,0,0,0,0,0.0,"""Weston,WY"""
2025,56,"""WY""",45,"""Weston """,9999,"""All Other Crops """,1,"""YP ""","""A ""","""RBUP""",0.7,1,0,0,0,0,"""Acres """,0,0,0,0,0,0,0,0,0,0.0,"""Weston,WY"""


In [59]:
## Filtering the location in the insurance data to match with weather data
common_locations = lagged_weather.select("location").unique().to_series().to_list()
common_locations

['Webster_County_IA',
 'Renville_County_MN',
 'Champaign_County_IL',
 'McLean_County_IL',
 'Dawson_County_NE',
 'Pottawattamie_County_IA',
 'Livingston_County_IL',
 'LaSalle_County_IL',
 'Saline_County_MO',
 'Iroquois_County_IL',
 'Story_County_IA']

In [60]:
list_locations = []
for i in common_locations:
    i = re.sub("_", "",i)
    i = re.sub(r'County',r",",i)
    list_locations.append(i)


In [61]:
list_locations

['Webster,IA',
 'Renville,MN',
 'Champaign,IL',
 'McLean,IL',
 'Dawson,NE',
 'Pottawattamie,IA',
 'Livingston,IL',
 'LaSalle,IL',
 'Saline,MO',
 'Iroquois,IL',
 'Story,IA']

In [62]:
insurance_df.columns

['commodity_year',
 'state_code',
 'state_abbreviation',
 'county_code',
 'county_name',
 'commodity_code',
 'commodity_name',
 'insurance_plan_code',
 'insurance_plan_abbreviation',
 'coverage_category',
 'delivery_type',
 'coverage_level',
 'policies_sold_count',
 'policies_earning_premium_count',
 'policies_indemnified_count',
 'units_earning_premium_count',
 'units_indemnified_count',
 'quantity_type',
 'net_reported_quantity',
 'endorsed_companion_acres',
 'liability_amount',
 'total_premium_amount',
 'subsidy_amount',
 'state_private_subsidy',
 'additional_subsidy',
 'efa_premium_discount',
 'indemnity_amount',
 'loss_ratio',
 'county_state']

In [63]:
## Filtering the location in the insurance data to match with weather data
filtered_insurance = (
    insurance_df
    .filter(pl.col("county_state").is_in(list_locations))
    .with_columns(pl.col("commodity_name").str.strip_chars(" ").alias("commodity_name"))
    .filter(pl.col("commodity_name") == "Corn")
    .select(['commodity_year', 'loss_ratio', 'county_state','net_reported_quantity'])
)
filtered_insurance.head()

commodity_year,loss_ratio,county_state,net_reported_quantity
i64,f64,str,i64
2014,3.09,"""Pottawattamie,IA""",91
2014,0.0,"""Pottawattamie,IA""",12
2014,4.74,"""Pottawattamie,IA""",1049
2014,2.87,"""Pottawattamie,IA""",3923
2014,4.42,"""Pottawattamie,IA""",23697


In [64]:
filtered_insurance.shape

(2144, 4)

In [65]:
fixed_lagged_weather = lagged_weather.with_columns(pl.col("location").str.replace("County", ",").str.replace_all("_", ""))
fixed_lagged_weather

location,year,month,ppt_total,rainy_days,ppt_variance,tmax,tmin,temp_range,tmax30,tmax35,tmin0,gdd,wind_mean,wind_max,wind_variance,high_wind_days,solar_mean,solar_sum,solar_variance,low_light_days,rh_mean,rh_min,rh_max,high_humidity_days,low_humidity_days,heat_humidity_stress_days,drought_stress_days,storm_potential_days,cloudy_warm_days,ppt_intensity,vpd_mean,ppt_total_AN,ppt_intensity_AN,ppt_variance_AN,tmax_AN,tmin_AN,…,tmin0_AN_lag2,wind_mean_AN_lag1,wind_mean_AN_lag2,wind_max_AN_lag1,wind_max_AN_lag2,wind_variance_AN_lag1,wind_variance_AN_lag2,high_wind_days_AN_lag1,high_wind_days_AN_lag2,solar_mean_AN_lag1,solar_mean_AN_lag2,solar_sum_AN_lag1,solar_sum_AN_lag2,solar_variance_AN_lag1,solar_variance_AN_lag2,low_light_days_AN_lag1,low_light_days_AN_lag2,rh_mean_AN_lag1,rh_mean_AN_lag2,rh_min_AN_lag1,rh_min_AN_lag2,rh_max_AN_lag1,rh_max_AN_lag2,high_humidity_days_AN_lag1,high_humidity_days_AN_lag2,low_humidity_days_AN_lag1,low_humidity_days_AN_lag2,heat_humidity_stress_days_AN_lag1,heat_humidity_stress_days_AN_lag2,drought_stress_days_AN_lag1,drought_stress_days_AN_lag2,storm_potential_days_AN_lag1,storm_potential_days_AN_lag2,cloudy_warm_days_AN_lag1,cloudy_warm_days_AN_lag2,vpd_mean_AN_lag1,vpd_mean_AN_lag2
str,i32,i8,f64,u32,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Pottawattamie,IA""",2014,7,66.07,8,7.741127,27.037097,15.549677,11.487419,7,0,0,350.095,3.809032,6.76,1.514533,0,23.087419,715.71,4.509607,31,73.316129,58.37,89.59,0,0,7,0,0,31,8.25875,0.675683,-0.635174,0.502122,1.405602,-1.684229,-2.411544,…,0.113067,0.891925,0.79576,-0.526612,-0.614319,0.308559,-0.550638,-0.301511,-0.661107,-0.359371,-0.807768,-0.359371,-0.877266,0.520184,0.759369,0.0,-0.583874,0.690159,-0.522651,0.971076,-0.959313,0.540556,-0.146763,-0.175382,-0.343797,0.0,0.0,0.0,0.0,-0.301511,0.0,-0.301511,0.0,-1.411588,-0.301511,-0.896625,0.56957
"""Renville,MN""",2016,11,38.15,8,2.957666,10.483667,0.480667,10.003,0,0,14,13.74,4.890333,12.55,2.163842,1,7.528667,225.86,3.018156,30,80.379667,70.53,95.58,6,0,0,0,0,2,4.76875,0.176997,0.253707,-0.283365,0.139449,1.306462,1.407077,…,0.0,-0.527662,0.367306,-0.74176,0.766814,-0.890692,1.061671,-0.301511,0.80403,0.736676,-0.441418,0.736676,-0.441418,0.717877,0.314519,0.0,0.0,-1.480872,-0.689707,-1.167655,-0.678929,-1.295709,-0.699574,0.0,-0.771421,0.0,0.0,0.94388,0.0,0.0,0.0,-0.661107,0.0,-0.330902,0.0,0.758847,0.624329
"""Pottawattamie,IA""",2018,4,20.21,3,1.678066,12.547333,-0.508333,13.055667,0,0,17,32.545,5.633667,11.23,2.251984,2,17.227667,516.83,7.79867,30,67.539,52.14,86.11,0,0,0,0,0,6,6.736667,0.303956,-1.253077,-0.164464,-1.103373,-2.445937,-2.269323,…,1.971679,0.796204,0.891925,0.472567,-0.526612,1.32964,0.308559,0.0,-0.301511,-0.346961,-0.359371,-0.346961,-0.359371,-1.23915,0.520184,0.0,0.0,0.53126,0.690159,0.342002,0.971076,0.661594,0.540556,-0.449467,-0.175382,0.0,0.0,0.754434,0.0,0.0,-0.301511,-0.301511,-0.301511,0.301511,-1.411588,-0.987505,-0.896625
"""Champaign,IL""",2019,9,82.1,10,4.629088,28.643333,16.833333,11.81,12,0,0,382.15,3.831667,6.69,1.318257,0,15.768667,473.06,4.794204,30,72.775667,57.51,86.82,0,0,7,0,0,30,8.21,0.752829,0.319939,-0.377972,-0.645944,1.189974,1.744997,…,-0.476731,-0.813278,0.410277,-1.85685,0.49994,-1.587781,1.33175,0.0,-0.449467,1.816255,-1.321049,1.816255,-1.321049,-1.686762,1.019129,0.0,0.0,-0.141175,1.692854,-0.850143,0.62701,0.206034,0.68209,0.0,0.568149,0.0,0.0,-0.393369,0.0,0.0,0.0,0.0,1.556998,0.0,0.028339,-0.088304,-0.87681
"""Story,IA""",2018,5,102.11,15,5.211043,26.19871,14.191935,12.006774,6,2,0,316.055,4.009032,7.11,1.397892,0,20.592903,638.38,7.684696,31,73.282903,55.05,89.65,0,0,2,0,0,28,6.807333,0.632314,-0.215643,-0.753367,-0.336808,2.042113,2.489856,…,0.0,1.18009,-1.520694,-0.076793,-1.554092,0.409892,-0.608428,0.0,0

In [66]:
fixed_lagged_weather.write_csv("C:\\Users\\Arnold\\OneDrive\\Desktop\\CAPSTONE PROJECT\\farming_risk_regions\\data\\interim\\weather_data\\lagged_weather_data.csv")

In [67]:
# Merge helper: normalize keys and join monthly (yearly insurance broadcast across months)

def _normalize_weather_key(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("location")
          .str.to_uppercase()
          .str.replace_all("COUNTY", ",")
          .str.replace_all(r"\\s+", "")   # remove spaces
          .str.replace_all("_", "")         # remove underscores
          .alias("join_key"),
        pl.col("year").cast(pl.Int64)
    ])


def _normalize_insurance_key(df: pl.DataFrame) -> pl.DataFrame:
    base = df
    # Ensure year column present and standardized name
    if "year" not in base.columns:
        base = base.with_columns(pl.col("commodity_year").cast(pl.Int64).alias("year"))
    return base.with_columns([
        pl.col("county_state")
          .str.to_uppercase()
          .str.replace_all(r"\\s+", "")   # 'Story, IA' -> 'STORY,IA'
          .alias("join_key"),
        pl.col("year").cast(pl.Int64)
    ])


def merge_and_prepare_polars(
    weather_df: pl.DataFrame,
    insurance_df: pl.DataFrame,
    join_grain: str = "monthly",   # "monthly" or "annual"
    drop_unmatched: bool = False,
) -> pl.DataFrame:
    """
    Merge insurance and weather data with proper temporal alignment using Polars.

    - monthly: broadcast yearly insurance values across all months of that year per location.
    - annual: aggregate weather to annual per location prior to merge.
    """
    w = _normalize_weather_key(weather_df)
    i = _normalize_insurance_key(insurance_df)

    if join_grain == "annual":
        # Aggregate weather to annual per join_key/year
        # Choose sensible aggregations: sums for totals, means for rates/indices
        sum_cols = [c for c in w.columns if c in {"ppt_total", "gdd", "solar_sum"}]
        # Aggregate all numeric columns by mean unless in sum_cols or are identifiers
        id_cols = {"join_key", "year", "location"}
        num_cols = [c for c, dt in zip(w.columns, w.dtypes) if c not in id_cols and pl.datatypes.is_numeric(dt)]
        mean_cols = [c for c in num_cols if c not in sum_cols]

        agg_exprs = [pl.col(c).sum().alias(c) for c in sum_cols] + [pl.col(c).mean().alias(c) for c in mean_cols]
        w_year = w.group_by(["join_key", "year"]).agg(agg_exprs)
        merged = w_year.join(i, on=["join_key", "year"], how="inner" if drop_unmatched else "left")
    else:
        # Monthly: keep monthly weather; join insurance on key+year (replicate values across months)
        merged = w.join(i, on=["join_key", "year"], how="inner" if drop_unmatched else "left")

    return merged


In [68]:

merged_dataset = merge_and_prepare_polars(fixed_lagged_weather, filtered_insurance, join_grain="monthly")
merged_dataset = merged_dataset.unique()
print(merged_dataset.shape)


(20178, 121)


In [69]:

merged_dataset.tail()

location,year,month,ppt_total,rainy_days,ppt_variance,tmax,tmin,temp_range,tmax30,tmax35,tmin0,gdd,wind_mean,wind_max,wind_variance,high_wind_days,solar_mean,solar_sum,solar_variance,low_light_days,rh_mean,rh_min,rh_max,high_humidity_days,low_humidity_days,heat_humidity_stress_days,drought_stress_days,storm_potential_days,cloudy_warm_days,ppt_intensity,vpd_mean,ppt_total_AN,ppt_intensity_AN,ppt_variance_AN,tmax_AN,tmin_AN,…,wind_variance_AN_lag1,wind_variance_AN_lag2,high_wind_days_AN_lag1,high_wind_days_AN_lag2,solar_mean_AN_lag1,solar_mean_AN_lag2,solar_sum_AN_lag1,solar_sum_AN_lag2,solar_variance_AN_lag1,solar_variance_AN_lag2,low_light_days_AN_lag1,low_light_days_AN_lag2,rh_mean_AN_lag1,rh_mean_AN_lag2,rh_min_AN_lag1,rh_min_AN_lag2,rh_max_AN_lag1,rh_max_AN_lag2,high_humidity_days_AN_lag1,high_humidity_days_AN_lag2,low_humidity_days_AN_lag1,low_humidity_days_AN_lag2,heat_humidity_stress_days_AN_lag1,heat_humidity_stress_days_AN_lag2,drought_stress_days_AN_lag1,drought_stress_days_AN_lag2,storm_potential_days_AN_lag1,storm_potential_days_AN_lag2,cloudy_warm_days_AN_lag1,cloudy_warm_days_AN_lag2,vpd_mean_AN_lag1,vpd_mean_AN_lag2,join_key,commodity_year,loss_ratio,county_state,net_reported_quantity
str,i64,i8,f64,u32,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,f64,str,i64
"""Pottawattamie,IA""",2018,8,177.0,13,14.389145,29.366452,18.040968,11.325484,17,2,0,424.815,3.877419,9.03,1.763272,0,18.210645,564.53,7.143088,31,72.25,50.02,91.92,2,0,5,0,1,31,13.615385,0.813425,1.013452,1.480958,2.069256,-0.200879,0.086751,…,1.956075,0.926058,-0.301511,0.0,-1.289119,-0.521125,-1.289119,-0.521125,0.804407,1.07992,0.0,0.0,0.61543,0.91429,0.489819,1.043334,0.547646,0.844925,1.11075,0.738112,0.0,0.0,0.0,-0.147784,-0.301511,-0.301511,-0.301511,-0.301511,-0.075892,0.0,-0.554147,-1.018715,"""POTTAWATTAMIE,IA""",2018,0.0,"""Pottawattamie,IA""",294
"""Livingston,IL""",2018,11,60.1,11,3.937279,4.781667,-3.211,7.992667,0,0,22,0.0,4.760333,8.67,1.868497,0,6.994,209.82,2.676433,30,82.927333,63.99,94.62,7,0,0,0,1,0,5.463636,0.110393,0.31169,-1.040515,-0.183472,-1.577894,-1.104187,…,0.293967,-0.006453,0.208063,0.0,1.215021,1.110261,1.215021,1.110261,-0.449814,-0.54731,0.0,0.0,-0.429823,-0.634876,-0.682121,-0.744289,-1.127817,-1.110539,-0.995593,-0.833476,0.0,0.0,0.0,1.111721,0.0,0.0,-0.301511,0.0,0.0,0.569552,0.273957,0.759595,"""LIVINGSTON,IL""",2018,0.0,"""Livingston,IL""",37
"""Iroquois,IL""",2016,10,67.63,7,4.324835,19.674516,8.516774,11.157742,0,0,0,133.555,4.610968,9.89,1.703249,0,10.83,335.73,3.848907,31,80.141613,67.33,90.15,1,0,0,0,0,16,9.661429,0.319433,-0.53142,0.191942,-0.631802,0.557136,0.922345,…,0.387439,-0.898155,0.346873,0.0,-1.281084,0.372933,-1.281084,0.372933,0.29215,-1.341307,0.0,0.0,1.286532,0.798315,0.801653,1.107589,-1.001171,0.033235,1.559087,0.72075,0.0,0.0,0.0,0.0,0.0,0.0,3.015113,-0.449467,0.0,-0.978492,-0.200847,-1.036025,"""IROQUOIS,IL""",2016,0.01,"""Iroquois,IL""",2169
"""Champaign,IL""",2016,4,96.43,9,5.618014,17.556333,4.865333,12.691,0,0,8,98.45,5.175333,9.5,1.940026,0,17.132667,513.98,7.129714,30,75.455,60.73,93.06,2,0,0,0,0,11,10.714444,0.326749,-0.066609,0.871617,-0.229044,0.180073,0.067293,…,-1.539314,1.372286,-0.449467,0.0,1.40561,0.140872,1.40561,0.140872,-0.520811,0.129931,0.0,0.0,-0.277038,0.78848,0.337909,0.503558,0.89151,0.838722,-0.324657,0.845154,0.0,0.0,0.0,2.0406,0.0,0.0,-0.583874,0.0,0.028339,0.0,0.239416,-0.589554,"""CHAMPAIGN,IL""",2016,0.0,"""Champaign,IL""",169
"""Livingston,IL""",2019,5,191.09,21,6.435333,21.356129,10.585484,10.770645,0,0,0,187.205,4.719355,9.25,1.87791,0,18.474839,572.72,5.888485,31,81.265806,62.91,92.21,1,0,0,0,0,20,9.099524,0.340007,2.160988,1.233809,0.233083,-0.66407,-0.184632,…,-0.689706,-1.118764,0.0,-0.936282,-0.481451,-1.545414,-0.481451,-1.545414,1.029272,

In [70]:
# Create loss column for modeling
# loss_ratio is the target: it represents indemnities/liabilities (loss severity)
# We'll use it directly as 'loss' since it's already a normalized measure

merged_dataset = merged_dataset.with_columns([
    pl.col("loss_ratio").alias("loss")
])

# Drop rows where loss is null (no insurance data available)
merged_dataset = merged_dataset.drop_nulls(subset=["loss"])

print(f"Final dataset shape: {merged_dataset.shape}")
print(f"\nLoss statistics:")
print(f"  Zero losses: {(merged_dataset['loss'] == 0).sum()} ({(merged_dataset['loss'] == 0).sum() / merged_dataset.height * 100:.1f}%)")
print(f"  Non-zero losses: {(merged_dataset['loss'] > 0).sum()} ({(merged_dataset['loss'] > 0).sum() / merged_dataset.height * 100:.1f}%)")
print(f"  Mean loss: {merged_dataset['loss'].mean():.4f}")
print(f"  Median loss: {merged_dataset['loss'].median():.4f}")
print(f"  Max loss: {merged_dataset['loss'].max():.4f}")

merged_dataset.head()

Final dataset shape: (19811, 122)

Loss statistics:
  Zero losses: 11160 (56.3%)
  Non-zero losses: 8651 (43.7%)
  Mean loss: 0.6640
  Median loss: 0.0000
  Max loss: 79.8400


location,year,month,ppt_total,rainy_days,ppt_variance,tmax,tmin,temp_range,tmax30,tmax35,tmin0,gdd,wind_mean,wind_max,wind_variance,high_wind_days,solar_mean,solar_sum,solar_variance,low_light_days,rh_mean,rh_min,rh_max,high_humidity_days,low_humidity_days,heat_humidity_stress_days,drought_stress_days,storm_potential_days,cloudy_warm_days,ppt_intensity,vpd_mean,ppt_total_AN,ppt_intensity_AN,ppt_variance_AN,tmax_AN,tmin_AN,…,wind_variance_AN_lag2,high_wind_days_AN_lag1,high_wind_days_AN_lag2,solar_mean_AN_lag1,solar_mean_AN_lag2,solar_sum_AN_lag1,solar_sum_AN_lag2,solar_variance_AN_lag1,solar_variance_AN_lag2,low_light_days_AN_lag1,low_light_days_AN_lag2,rh_mean_AN_lag1,rh_mean_AN_lag2,rh_min_AN_lag1,rh_min_AN_lag2,rh_max_AN_lag1,rh_max_AN_lag2,high_humidity_days_AN_lag1,high_humidity_days_AN_lag2,low_humidity_days_AN_lag1,low_humidity_days_AN_lag2,heat_humidity_stress_days_AN_lag1,heat_humidity_stress_days_AN_lag2,drought_stress_days_AN_lag1,drought_stress_days_AN_lag2,storm_potential_days_AN_lag1,storm_potential_days_AN_lag2,cloudy_warm_days_AN_lag1,cloudy_warm_days_AN_lag2,vpd_mean_AN_lag1,vpd_mean_AN_lag2,join_key,commodity_year,loss_ratio,county_state,net_reported_quantity,loss
str,i64,i8,f64,u32,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,f64,str,i64,f64
"""Saline,MO""",2020,7,160.79,17,6.944042,30.130323,21.212581,8.917742,19,0,0,485.815,3.003871,5.49,0.99781,0,22.322258,691.99,6.132272,31,82.309677,67.97,92.14,3,0,18,0,0,31,9.458235,0.583198,0.629495,-0.175785,-0.223165,-0.490424,0.703775,…,-1.471147,0.0,-0.870388,0.770858,0.921196,0.770858,0.921196,0.038246,-0.219322,0.0,0.0,0.007906,-1.021617,0.20688,-0.474761,-0.911229,-0.764992,-0.924995,-1.093697,0.0,0.0,0.16855,0.0,0.0,0.0,-0.301511,-0.583874,1.343927,1.580349,0.205062,1.672847,"""SALINE,MO""",2020,0.0,"""Saline,MO""",2919,0.0
"""Champaign,IL""",2022,3,119.36,13,8.82349,11.488387,-0.298387,11.786774,0,0,17,23.31,5.866774,9.2,1.91265,0,13.129032,407.0,5.490026,31,77.57129,61.68,92.9,3,0,0,0,1,5,9.181538,0.203924,1.202192,0.200763,1.625074,0.230879,0.124804,…,0.694921,0.0,0.0,-0.255115,0.445665,-0.255115,0.445665,0.890702,0.676537,0.0,0.0,0.295784,-0.013447,0.430827,-0.294807,0.576559,0.187712,0.88991,-0.069911,0.0,0.0,1.124643,0.597614,0.0,0.0,2.0226,0.0,-0.455289,0.0,-0.527178,-0.08485,"""CHAMPAIGN,IL""",2022,0.0,"""Champaign,IL""",2876,0.0
"""Pottawattamie,IA""",2024,3,53.18,6,4.047989,12.017742,-0.743548,12.76129,0,0,17,10.185,5.974194,9.72,1.873307,0,14.437419,447.56,5.730397,31,64.952258,41.6,90.14,1,0,0,0,0,1,8.863333,0.319591,0.161609,0.261125,0.065481,0.755391,0.692802,…,0.050005,2.247765,0.0,-0.016022,0.486568,0.270426,0.486568,-0.94766,-1.874261,1.556998,0.0,0.8605,-0.900086,0.976357,-0.307286,0.211992,-0.804652,0.286498,-0.301511,0.0,0.0,0.0,-0.527645,0.0,0.0,0.0,-0.661107,-0.301511,0.08558,-0.247906,0.525287,"""POTTAWATTAMIE,IA""",2024,0.0,"""Pottawattamie,IA""",188,0.0
"""Renville,MN""",2021,9,90.38,7,7.188468,24.939333,12.221,12.718333,2,0,0,257.405,4.686333,8.73,1.650142,0,15.976333,479.29,5.272711,30,66.202667,46.76,86.68,0,0,0,0,0,27,12.911429,0.72343,0.428036,1.144739,0.661625,0.228383,-0.048975,…,0.554178,1.909572,0.0,1.711697,-0.929096,1.711697,-0.929096,-1.157089,0.616855,0.0,0.0,-1.375715,1.728156,-0.486848,1.879906,0.879243,0.885344,-1.180367,1.868397,0.0,0.0,0.0,-0.699113,0.0,-0.301511,0.0,0.0,0.0,-0.61744,0.325262,-1.44002,"""RENVILLE,MN""",2021,0.25,"""Renville,MN""",57755,0.25
"""Iroquois,IL""",2022,6,54.73,9,4.398448,27.977333,15.37,12.607333,10,0,0,350.21,3.870667,6.4,1.116593,0,23.803667,714.11,7.275124,30,68.345667,49.58,84.81,0,0,5,0,0,30,6.081111,0.820416,-1.101261,-1.229692,-0.921875,0.718772,-0.705558,…,-0.95229,0.72075,0.0,0.525753,-0.059159,0.525753,-0.059159,1.537475,0.534495,0.0,0.0,0.038395,0.246019,-0.698361,0.

In [72]:
# Save the final merged dataset for modeling (as pandas-compatible CSV for the existing modeling code)
# Convert to pandas for compatibility with the existing modeling.ipynb
output_path = "C:\\Users\\Arnold\\OneDrive\\Desktop\\CAPSTONE PROJECT\\farming_risk_regions\\data\\processed\\merged_weather_insurance_data.csv"

# Convert Polars to pandas for the modeling notebook
merged_dataset_pd = merged_dataset.to_pandas()
merged_dataset_pd.to_csv(output_path, index=False)

print(f"Saved merged dataset to: {output_path}")
print(f"  Shape: {merged_dataset_pd.shape}")
print(f"  Columns: {list(merged_dataset_pd.columns[:10])}... ({len(merged_dataset_pd.columns)} total)")

Saved merged dataset to: C:\Users\Arnold\OneDrive\Desktop\CAPSTONE PROJECT\farming_risk_regions\data\processed\merged_weather_insurance_data.csv
  Shape: (19811, 122)
  Columns: ['location', 'year', 'month', 'ppt_total', 'rainy_days', 'ppt_variance', 'tmax', 'tmin', 'temp_range', 'tmax30']... (122 total)
